# UK Road Accident Analysis (2005–2014)

**Dataset:** UK Road Safety Data — 10 years of reported road accidents  
**Objective:** Identify key drivers of accident severity (Fatal / Serious / Slight) through a structured EDA pipeline  

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Data Cleaning — nulls, coded unknowns, datetime parsing, label mapping |
| 2 | Univariate Analysis — distribution of each key variable |
| 3 | Bivariate Analysis — severity vs environmental/temporal factors |
| 4 | Temporal Heatmaps — Hour × Day accident density |
| 5 | Multivariate Analysis — combined factor interactions |
| 6 | Correlation & Feature Importance — RF + chi-square |
| 7 | Geographic Analysis — district and police force breakdown |
| 8 | Deep Dives — high-risk scenario profiling |
| 9 | Final Summary — headline KPIs |

---

> **Note:** Charts are saved to `OUTPUT` directory. Adjust `OUTPUT` and CSV path before running.

## Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # Non-interactive backend — swap to 'inline' in Jupyter if preferred
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Paths ────────────────────────────────────────────────────────────────────
CSV_PATH = '../Data/Raw/UK_Accident.csv'          # update if needed
OUTPUT   = '../UK_Accident_Analyzed/'
os.makedirs(OUTPUT, exist_ok=True)

# ── Helper utilities ─────────────────────────────────────────────────────────

def save(fig, name):
    """Save a matplotlib figure to the OUTPUT directory and close it."""
    fig.savefig(f'{OUTPUT}{name}.png', bbox_inches='tight', dpi=120)
    plt.close(fig)

---
## Step 1 — Data Loading

In [54]:
# Load CSV; index_col=0 drops the unnamed index column exported by some tools.
# low_memory=False prevents mixed-type inference warnings on large files.
df = pd.read_csv(CSV_PATH, index_col=0, low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

Columns: ['Accident_Index', 'Location_Easting_OSGR', 'Location_Northing_OSGR', 'Longitude', 'Latitude', 'Police_Force', 'Accident_Severity', 'Number_of_Vehicles', 'Number_of_Casualties', 'Date', 'Day_of_Week', 'Time', 'Local_Authority_(District)', 'Local_Authority_(Highway)', '1st_Road_Class', '1st_Road_Number', 'Road_Type', 'Speed_limit', 'Junction_Control', '2nd_Road_Class', '2nd_Road_Number', 'Pedestrian_Crossing-Human_Control', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Special_Conditions_at_Site', 'Carriageway_Hazards', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident', 'LSOA_of_Accident_Location', 'Year']


---
## Step 2 — Data Cleaning

Cleaning actions performed:
- **Missing value audit** — count and percentage per column
- **Coded unknowns** — the dataset uses `-1` for "Data missing or out of range" in categorical fields; these are replaced with `NaN`
- **Datetime parsing** — extract `Month`, `Month_Name`, and `Hour` from raw date/time strings
- **Label mapping** — decode numeric codes for `Severity`, `Day_of_Week`, and `Urban_or_Rural_Area` into human-readable strings
- **Column pruning** — drop columns with >95% missing values and redundant geospatial ID columns
- **Deduplication** — remove duplicate `Accident_Index` rows

In [55]:
# ── 2a. Missing value audit ───────────────────────────────────────────────────
nulls    = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({'null_count': nulls, 'null_pct': null_pct})
null_report = null_report[null_report['null_count'] > 0].sort_values('null_pct', ascending=False)
print(null_report)

                                         null_count  null_pct
Carriageway_Hazards                         1476900     98.19
Special_Conditions_at_Site                  1467568     97.57
Junction_Control                             602835     40.08
LSOA_of_Accident_Location                    108238      7.20
Location_Easting_OSGR                           101      0.01
Longitude                                       101      0.01
Time                                            117      0.01
Pedestrian_Crossing-Human_Control                17      0.00
Pedestrian_Crossing-Physical_Facilities          34      0.00


In [56]:
# ── 2b. Coded unknowns (-1 sentinel values) ──────────────────────────────────
# The DfT encoding uses -1 to signal "Data missing / out of range" in these fields.
cat_cols = [
    'Junction_Control', '2nd_Road_Class',
    'Pedestrian_Crossing-Human_Control', 'Pedestrian_Crossing-Physical_Facilities',
    'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions',
    'Special_Conditions_at_Site', 'Carriageway_Hazards'
]
for col in cat_cols:
    if col in df.columns:
        neg = (df[col] == -1).sum()
        if neg > 0:
            print(f'  {col}: {neg} rows with -1 ({neg/len(df)*100:.2f}%)')

  2nd_Road_Class: 615474 rows with -1 (40.92%)


In [57]:
# ── 2c. Parse datetime fields ─────────────────────────────────────────────────
# Date column uses UK day-first format (DD/MM/YYYY)
df['Date']       = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
# Time stored as 'HH:MM' string — extract the hour component only
df['Hour']       = df['Time'].str.split(':').str[0].astype(float)

# ── 2d. Decode severity codes ─────────────────────────────────────────────────
# DfT codes: 1=Fatal, 2=Serious, 3=Slight
severity_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
df['Severity_Label'] = df['Accident_Severity'].map(severity_map)

# ── 2e. Decode day-of-week codes ─────────────────────────────────────────────
# DfT codes: 1=Sunday … 7=Saturday
day_map = {1:'Sunday',2:'Monday',3:'Tuesday',4:'Wednesday',5:'Thursday',6:'Friday',7:'Saturday'}
df['Day_Name'] = df['Day_of_Week'].map(day_map)

# ── 2f. Decode urban/rural codes ─────────────────────────────────────────────
df['Area_Type'] = df['Urban_or_Rural_Area'].map({1:'Urban', 2:'Rural', 3:'Unallocated'})

In [58]:
# ── 2g. Replace -1 sentinels with NaN in categorical columns ─────────────────
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].replace(-1, np.nan)

# ── Drop near-entirely-null columns (>95% missing) ───────────────────────────
# 'Carriageway_Hazards' and 'Special_Conditions_at_Site' are mostly blank
df.drop(columns=['Carriageway_Hazards', 'Special_Conditions_at_Site'], inplace=True, errors='ignore')

# ── 2h. Drop redundant geospatial ID / coordinate columns ────────────────────
# LSOA and OSGR columns are superseded by Lat/Long and add no analytical value
drop_cols = ['LSOA_of_Accident_Location', 'Location_Easting_OSGR', 'Location_Northing_OSGR']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# ── 2i. Deduplicate on Accident_Index ────────────────────────────────────────
dups = df.duplicated(subset='Accident_Index').sum()
print(f'\nDuplicate Accident_Index rows: {dups}')
df.drop_duplicates(subset='Accident_Index', inplace=True)

print(f'\nClean shape : {df.shape}')
print(f'Date range  : {df["Date"].min()} → {df["Date"].max()}')
print(f'Years present: {sorted(df["Year"].unique())}')

Duplicate Accident_Index rows: 576763

Clean shape : (927387, 33)
Date range  : 2005-01-01 00:00:00 → 2014-12-31 00:00:00
Years present: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014)]


### 2j–2k. Additional Cleaning (from cleaning notebook)

- **String placeholders → NaN** — text-encoded unknowns (`'Unknown'`, `'Not known'`, `'Data missing or out of range'`, `'Not available'`) replaced with `NaN`
- **Lat/Long range validation** — nullify coordinates outside UK bounding box (`lat 49–61`, `lon -9–2`)
- **Speed limit validation** — nullify values outside `[10, 80]` mph
- **Negative value guard** — nullify negative casualty/vehicle counts

In [59]:
# ── 2j. String placeholder → NaN (text-encoded unknowns) ─────────────────────
# Some fields use string values instead of -1 to denote missing/unknown data
string_placeholders = ['Data missing or out of range', 'Unknown', 'Not known', 'Not available']
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].replace(string_placeholders, np.nan)

# ── 2k. Range validation ──────────────────────────────────────────────────────
# Lat/Long: nullify anything outside UK bounding box
if 'Latitude' in df.columns:
    df.loc[~df['Latitude'].between(49, 61), 'Latitude'] = np.nan
if 'Longitude' in df.columns:
    df.loc[~df['Longitude'].between(-9, 2), 'Longitude'] = np.nan

# Speed limit: nullify physically impossible values
if 'Speed_limit' in df.columns:
    df.loc[~df['Speed_limit'].between(10, 80), 'Speed_limit'] = np.nan

# Casualty / vehicle counts: nullify negatives
for col in ['Number_of_Casualties', 'Number_of_Vehicles']:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan

print('Range validation done.')
print(f'Null Latitude  : {df["Latitude"].isna().sum() if "Latitude" in df.columns else "N/A"}')
print(f'Null Longitude : {df["Longitude"].isna().sum() if "Longitude" in df.columns else "N/A"}')
print(f'Null Speed_limit: {df["Speed_limit"].isna().sum() if "Speed_limit" in df.columns else "N/A"}')

Range validation done.
Null Latitude  : 57
Null Longitude : 57
Null Speed_limit: 0


In [ ]:
# ── Export Cleaned Data ───────────────────────────────────────────────────────
export_path = '../UK_Accident_cleaned.csv'
df.to_csv(export_path, index=False)
print(f'Successfully exported cleaned data to {export_path}')

In [98]:
df.isnull().sum()

Accident_Index                                      0
Longitude                                          57
Latitude                                           57
Police_Force                                        0
Accident_Severity                                   0
Number_of_Vehicles                                  0
Number_of_Casualties                                0
Date                                                0
Day_of_Week                                         0
Time                                               10
Local_Authority_(District)                          0
Local_Authority_(Highway)                           0
1st_Road_Class                                      0
1st_Road_Number                                     0
Road_Type                                        5095
Speed_limit                                         0
Junction_Control                               351501
2nd_Road_Class                                 357890
2nd_Road_Number             